# GENERACIÓN DE TEXTO CON REDES NEURONALES RECURRENTES

## 1 - El set de datos y el problema a resolver

Usaremos un texto en español, de libre acceso, y que contiene una serie de [cuentos de Edgar Allan Poe](https://www.gutenberg.org/cache/epub/46196/pg46196.txt).

La idea es crear un modelo de lenguaje a nivel de caracter: para cada predicción tomará 10 caracteres de entrada y generará 1 de salida:

![](https://drive.google.com/uc?export=view&id=1UqXixOnlp4AS7gL1EZiZW0J1qJ9ctg3B)

In [3]:
# Las librerías a utilizar
import numpy as np
from tensorflow.keras.layers import Dense, SimpleRNN
from keras.models import Sequential
from google.colab import drive
import tensorflow as tf
import requests

In [4]:
ruta = "https://raw.githubusercontent.com/cagomezv/AP_USA/main/Semana_3/data/cuentos_EAPoe.txt"

texto = requests.get(ruta).text

chars = list(set(texto))
tam_vocab = len(chars)

print(texto[:1000])  # Mostrar solo los primeros 1000 caracteres
print(chars)
print(tam_vocab)

EL BARRIL DE AMONTILLADO


HABÍA soportado lo mejor posible los mil pequeños agravios de Fortunato;
pero cuando se atrevió a llegar hasta el ultraje, juré que había de
vengarme. Vosotros, que tan bien conocéis mi temperamento, no supondréis
que pronuncié la más ligera amenaza. _Algún día_ me vengaría; esto era
definitivo; pero la misma decisión que abrigaba, excluía toda idea de
correr el menor riesgo. No solamente era necesario castigar, sino
castigar con impunidad. No se repara un agravio cuando la reparación se
vuelve en contra del justiciero; ni tampoco se repara cuando no se hace
sentir al ofensor de qué parte proviene el castigo.

Es necesario tener presente que jamás había dado a Fortunato, ni por
medio de palabras ni de acciones, ocasión de sospechar de mi buena
voluntad. Continué sonriéndole siempre, como era mi deseo, y él no se
apercibió de que _ahora_ sonreía yo al pensamiento de su inmolación.

Fortunato tenía un punto débil, aunque en otras cosas era hombre que
inspiraba 

## 2 - Creación del set de entrenamiento

In [5]:
# Diccionarios para convertir de texto a índice numérico y viceversa
char2ix = {c: i for i, c in enumerate(chars)}
ix2char = {i: c for i, c in enumerate(chars)}
print(char2ix)
print(ix2char)

{'â': 0, 'f': 1, 'r': 2, 'w': 3, 'J': 4, 'P': 5, '1': 6, 'æ': 7, 'y': 8, 'S': 9, 'Á': 10, 's': 11, 'W': 12, 'O': 13, 'p': 14, 'a': 15, '2': 16, 'l': 17, 'd': 18, '\n': 19, 'K': 20, 'H': 21, '¡': 22, 'j': 23, '9': 24, 'm': 25, ':': 26, '_': 27, '3': 28, '{': 29, '6': 30, 'T': 31, 'é': 32, 'ñ': 33, 'z': 34, 'M': 35, 'Í': 36, '¶': 37, '*': 38, '8': 39, 'ö': 40, 'R': 41, ')': 42, '¿': 43, ' ': 44, 'Q': 45, 'Ó': 46, '0': 47, '-': 48, '.': 49, 'U': 50, 'D': 51, 'v': 52, 'F': 53, 'á': 54, 'ô': 55, 'E': 56, 'B': 57, ']': 58, 'i': 59, 'L': 60, 'x': 61, 'h': 62, 'Y': 63, 'Ú': 64, "'": 65, 'N': 66, '7': 67, '!': 68, 'V': 69, 'í': 70, 'I': 71, 't': 72, 'A': 73, 'b': 74, 'g': 75, '?': 76, '(': 77, ',': 78, 'ú': 79, '4': 80, 'X': 81, 'G': 82, 'Ö': 83, 'c': 84, 'n': 85, ';': 86, 'k': 87, 'ü': 88, '"': 89, 'ó': 90, '5': 91, 'C': 92, '[': 93, 'e': 94, 'q': 95, 'u': 96, 'É': 97, 'o': 98, '}': 99}
{0: 'â', 1: 'f', 2: 'r', 3: 'w', 4: 'J', 5: 'P', 6: '1', 7: 'æ', 8: 'y', 9: 'S', 10: 'Á', 11: 's', 12: 'W', 

In [6]:
# Crear bloques de 10 caracteres, que serán los datos de entrada a la red
LONG_SEC = 10
texto_in, texto_out = [], []

# Por cada 10 caracteres de entrada, el modelo predice 1 caracter de salida
for i in range(0,len(texto)-LONG_SEC):
  texto_in.append(texto[i:i+LONG_SEC]) # Ejemplo: caracteres del 0 al 9
  texto_out.append(texto[i + LONG_SEC])# Ejemplo: caracter 10

print(texto_in[0])
print(texto_out[0])
print(texto_in[1])
print(texto_out[1])

EL BARRIL 
D
L BARRIL D
E


In [7]:
# Set de entrenamiento: simplemente se deben convertir a one-hot los sets
# de entrada y salida

X = np.zeros((len(texto_in), LONG_SEC, tam_vocab)) # #ejemplos x 10 x tam_vocab
Y = np.zeros((len(texto_in), tam_vocab))   # #ejemplos x tam_vocab

for i, entrada in enumerate(texto_in):
  for j, car in enumerate(entrada):
    X[i,j, char2ix[car]] = 1
    Y[i, char2ix[texto_out[i]]] = 1

print(X.shape)
print(Y.shape)
print(X[0,0,:])
print(Y[0,:])

(331185, 10, 100)
(331185, 100)
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]


## 3 - Crear modelo

In [9]:
SEED = 15
tf.random.set_seed(SEED)
np.random.seed(SEED)

N_NEURONAS = 256

modelo = Sequential()

modelo.add(
    SimpleRNN(
        N_NEURONAS,
        input_shape=(LONG_SEC, tam_vocab)
    )
)

modelo.add(
    Dense(
        tam_vocab,
        activation='softmax'
    )
)

modelo.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 256)            │        91,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        25,700 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,092 (457.39 KB)

 Trainable params: 117,092 (457.39 KB)

 Non-trainable params: 0 (0.00 B)

## 4 - Función para generación de texto

In [10]:
def generar_texto(modelo, L_OUT_SEC):
  # Seleccionar un dato de entrenamiento inicial aleatoriamente
  ix_test = np.random.randint(len(texto_in))
  char_test = texto_in[ix_test]

  # Generar texto de tamaño L_OUT_SEC
  print(char_test, end="")
  for i in range(L_OUT_SEC):
    # Codificación one-hot del texto de entrada
    X_test = np.zeros((1, LONG_SEC, tam_vocab)) # 1x10x100
    for j, car in enumerate(char_test):
      X_test[0,j, char2ix[car]] = 1

    # Introducir al modelo y generar predicción
    pred = modelo.predict(X_test, verbose=0)[0]
    y_pred = ix2char[np.argmax(pred)]

    # Imprimir el caracter predicho
    print(y_pred, end="")

    # Agregar como último elemento de "char_test" y continuar prediciendo
    char_test = char_test[1:] + y_pred
  print()

## 5 - Entrenamiento y generación de texto

In [11]:
modelo.compile(loss="categorical_crossentropy", optimizer="rmsprop")

NEPOCHS = 20
BATCH_SIZE = 128
L_OUT_SEC = 50      # Número de caracteres a generar

print('-'*50)
print('Antes del entrenamiento:')
generar_texto(modelo, L_OUT_SEC)

for it in range(NEPOCHS):
  modelo.fit(X,Y, batch_size=BATCH_SIZE, epochs=1, verbose=0)
  print('-'*50)
  print(f'Iteración: {it+1}')
  generar_texto(modelo, L_OUT_SEC)

--------------------------------------------------
Antes del entrenamiento:
aba y acar_:PéS.ñiCK(D)03Klíisóc-n¿7;üáIvYæVrs"fK¶3d¿z-
ÚBel
--------------------------------------------------
Iteración: 1
 ruin amista en es pera de la de la conte en es pera de la d
--------------------------------------------------
Iteración: 2
 doble Dupara de la por en es por el por en en su había es p
--------------------------------------------------
Iteración: 3
eza, mucho de la por el pero de la por el pero de la por el 
--------------------------------------------------
Iteración: 4
 pausada y sontra en la para de la para de la para de la par
--------------------------------------------------
Iteración: 5
s. Legrando de la para el propero de de la caraza de dista c
--------------------------------------------------
Iteración: 6
encontrában en la para resta de la presentada de algo de el 
--------------------------------------------------
Iteración: 7
respecto?----pero el temos al fin la para resta d